# Jour 6 — Curriculum & Robustesse

Ce notebook :
1. Démontre le curriculum d'entraînement (progression des stages)
2. Compare un agent baseline vs un agent entraîné avec curriculum
3. Analyse la robustesse sur des scénarios hors distribution (OOD)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import EnvConfig, ThermalConfig, RewardConfig
from envs.battery_thermal_env import BatteryThermalEnv
from envs.curriculum_env import CurriculumEnv, DEFAULT_STAGES
from evaluate import load_agent, run_episode
from robustness import _make_scenarios, _eval_scenario, _print_table

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Curriculum — visualisation des stages

In [ ]:
# Affiche les stages du curriculum par défaut
print('Curriculum par défaut :')
print(f'{"Stage":<8} {"Step min":>10}  {"T_amb_min":>10}  {"T_amb_max":>10}  {"I_max":>8}  {"I_min":>8}')
print('-' * 65)
for i, (step_min, cfg) in enumerate(DEFAULT_STAGES):
    print(f'{i:<8} {step_min:>10}  '
          f'{cfg["T_amb_min"]:>10.1f}  '
          f'{cfg["T_amb_max"]:>10.1f}  '
          f'{cfg["I_max"]:>8.1f}  '
          f'{cfg["I_min"]:>8.1f}')

In [ ]:
# Démonstration : montre que les paramètres changent à chaque stage
env_cfg = EnvConfig(thermal=ThermalConfig(max_steps=100), reward=RewardConfig())
base_env = BatteryThermalEnv(config=env_cfg)
cur_env  = CurriculumEnv(base_env)

obs, _ = cur_env.reset(seed=0)
stage_history = []

for step in range(200_000):
    action = cur_env.action_space.sample()
    _, _, terminated, truncated, info = cur_env.step(action)
    stage_history.append(info['curriculum_stage'])
    if terminated or truncated:
        cur_env.reset()
    if step % 40_000 == 0:
        tc = cur_env.env.tc
        print(f'  Step {step:>7} | stage={info["curriculum_stage"]} '
              f'| I_max={tc.I_max:.0f}A '
              f'| T_amb=[{tc.T_amb_min:.0f},{tc.T_amb_max:.0f}]°C')

print(f'\nTransitions enregistrées : {cur_env.stage_log}')

In [ ]:
# Plot de progression des stages
fig, ax = plt.subplots(figsize=(13, 3))
steps = range(len(stage_history))
ax.plot(steps, stage_history, color='tab:blue', linewidth=1.0)
ax.fill_between(steps, stage_history, alpha=0.2, color='tab:blue')
for step_t, stage in cur_env.stage_log:
    ax.axvline(step_t, color='red', linestyle='--', alpha=0.7)
    ax.text(step_t + 1000, stage - 0.1, f'Stage {stage}', fontsize=9, color='red')
ax.set_title('Progression du curriculum')
ax.set_xlabel('Steps')
ax.set_ylabel('Stage')
ax.set_yticks([0, 1, 2])
plt.tight_layout()
plt.show()

## 2. Test de robustesse — agent Jour 3

In [ ]:
CHECKPOINT  = '../runs/day3_run/checkpoints/sac_final.pt'
N_EPISODES  = 10

base_tc  = ThermalConfig()
env      = BatteryThermalEnv(config=EnvConfig(thermal=base_tc))
agent    = load_agent(
    CHECKPOINT,
    obs_dim    = env.observation_space.shape[0],
    action_dim = env.action_space.shape[0],
)
print('Agent chargé.')

In [ ]:
scenarios = _make_scenarios(base_tc)

print('Évaluation en cours...')
rows = []
for name, tc in scenarios.items():
    print(f'  {name:<16} ...', end=' ', flush=True)
    m = _eval_scenario(agent, name, tc, N_EPISODES)
    rows.append(m)
    print(f'return={m["return_mean"]:.1f}  pct_safe={m["pct_safe_mean"]:.1f}%')

_print_table(rows, rows[0]['return_mean'])

In [ ]:
# DataFrame pour analyse
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'results'} for r in rows])
df['degradation_%'] = 100 * (df['return_mean'] - df['return_mean'].iloc[0]) / abs(df['return_mean'].iloc[0])
print(df[['scenario', 'return_mean', 'return_std', 'pct_safe_mean', 'T_max_mean', 'degradation_%']].to_string(index=False))

In [ ]:
# Barplots robustesse
names     = [r['scenario']      for r in rows]
returns   = [r['return_mean']   for r in rows]
errs      = [r['return_std']    for r in rows]
pct_safes = [r['pct_safe_mean'] for r in rows]
colors    = ['tab:blue', 'tab:red', 'tab:cyan', 'tab:orange', 'tab:purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.bar(names, returns, yerr=errs, color=colors, capsize=5, alpha=0.85)
ax.axhline(returns[0], color='black', linestyle='--', linewidth=1, alpha=0.6, label='Baseline')
ax.set_title('Return moyen par scénario OOD', fontsize=12)
ax.set_ylabel('Return')
ax.tick_params(axis='x', rotation=20)
ax.legend(fontsize=9)

ax = axes[1]
bars = ax.bar(names, pct_safes, color=colors, alpha=0.85)
ax.axhline(100, color='green', linestyle='--', linewidth=0.8)
ax.axhline(pct_safes[0], color='black', linestyle='--', linewidth=1, alpha=0.6, label='Baseline')
ax.set_title('% Steps en zone sûre', fontsize=12)
ax.set_ylabel('% steps')
ax.set_ylim(0, 115)
ax.tick_params(axis='x', rotation=20)
ax.legend(fontsize=9)
for bar, val in zip(bars, pct_safes):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../runs/day3_run/checkpoints/robustness_metrics.png', dpi=120)
plt.show()

In [ ]:
# Trajectoires OOD
fig, ax = plt.subplots(figsize=(14, 6))

for row, color in zip(rows, colors):
    best = max(row['results'], key=lambda r: r['return'])
    ax.plot(best['T_hist'], label=row['scenario'], color=color, linewidth=1.3, alpha=0.85)

ax.axhline(base_tc.T_safe_min, color='blue',   linestyle='--', alpha=0.5,
           label=f'T_safe_min ({base_tc.T_safe_min}°C)')
ax.axhline(base_tc.T_safe_max, color='orange', linestyle='--', alpha=0.5,
           label=f'T_safe_max ({base_tc.T_safe_max}°C)')
ax.axhline(base_tc.T_cutoff,   color='red',    linestyle=':',  alpha=0.5,
           label=f'T_cutoff ({base_tc.T_cutoff}°C)')
ax.fill_between(range(max(len(r['results'][0]['T_hist']) for r in rows)),
                base_tc.T_safe_min, base_tc.T_safe_max, alpha=0.07, color='green')

ax.set_title('Trajectoires OOD — meilleur épisode par scénario', fontsize=12)
ax.set_xlabel('Step')
ax.set_ylabel('Température (°C)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Lancer un run avec curriculum (optionnel)

Pour entraîner un agent avec le curriculum et comparer :

```bash
python train.py --total_steps 200000 --run_name day6_curriculum --eval_every 20
```

Puis évaluer :
```bash
python robustness.py --checkpoint runs/day6_curriculum/checkpoints/sac_final.pt
```